# Pinecone Vector Search Demo (Offline, API-Key-Free)

This notebook demonstrates the **Pinecone Python client API shape** -- `create_index`, `Index`,
`upsert`, `query` -- used in the Virtual Liaison platform's project-data and cost-catalog RAG
pipelines (see `../02-vector-databases-and-pinecone.md`).

To run **fully offline with no API key**, we implement a `FakePineconeIndex` class that mirrors the
real `pinecone` client's method signatures using plain Python dicts and numpy for similarity search.
Everywhere the real client would be used is marked with a comment showing the real equivalent.

In [1]:
import numpy as np
import re

np.random.seed(42)
print("Ready.")

Ready.


## 1. A tiny hand-rolled "embedding" function

Real code would call an embedding model (`OpenAIEmbeddings`, etc.). Here we deterministically hash
words into a small fixed-dimension vector so the same text always produces the same vector, and
similar text (shared words) produces similar vectors -- good enough to demonstrate index/upsert/query
mechanics without any external dependency.

In [2]:
EMBED_DIM = 32

def fake_embed(text: str) -> list:
    """Deterministic bag-of-words hashing embedding -- a stand-in for a real embedding model call,
    e.g. OpenAIEmbeddings().embed_query(text)."""
    vec = np.zeros(EMBED_DIM)
    words = re.findall(r"[a-zA-Z0-9\-]+", text.lower())
    for w in words:
        idx = hash(w) % EMBED_DIM
        vec[idx] += 1.0
    norm = np.linalg.norm(vec)
    return (vec / norm if norm > 0 else vec).tolist()

fake_embed("Project Atlas Japan localization status")[:8]

[0.0, 0.0, 0.0, 0.0, 0.4472135954999579, 0.0, 0.0, 0.0]

## 2. `FakePineconeIndex` -- same method shapes as the real `pinecone` client

Real code:

```python
from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key="...")
pc.create_index(name="indegene-project-data", dimension=1536, metric="cosine",
                 spec=ServerlessSpec(cloud="aws", region="us-east-1"))
index = pc.Index("indegene-project-data")
```

Offline stand-in below: `FakePineconeIndex` implements `upsert(vectors=..., namespace=...)` and
`query(vector=..., top_k=..., namespace=..., filter=..., include_metadata=...)` with the identical
keyword-argument shape, backed by an in-memory dict instead of a managed service.

In [3]:
class FakePineconeIndex:
    """In-memory stand-in for pinecone.Index with the same method signatures.

    Swap this for the real thing by replacing:
        index = FakePineconeIndex(dimension=EMBED_DIM, metric="cosine")
    with:
        from pinecone import Pinecone
        pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
        index = pc.Index("indegene-project-data")
    Every call below (`.upsert(...)`, `.query(...)`) keeps working unchanged.
    """

    def __init__(self, dimension: int, metric: str = "cosine"):
        self.dimension = dimension
        self.metric = metric
        # namespace -> {id: {"values": [...], "metadata": {...}}}
        self._store: dict[str, dict[str, dict]] = {}

    def upsert(self, vectors: list[dict], namespace: str = "") -> dict:
        ns = self._store.setdefault(namespace, {})
        for v in vectors:
            ns[v["id"]] = {"values": v["values"], "metadata": v.get("metadata", {})}
        return {"upserted_count": len(vectors)}

    def _matches_filter(self, metadata: dict, filt: dict | None) -> bool:
        if not filt:
            return True
        for key, cond in filt.items():
            val = metadata.get(key)
            if isinstance(cond, dict):
                if "$eq" in cond and val != cond["$eq"]:
                    return False
                if "$in" in cond and val not in cond["$in"]:
                    return False
            else:
                if val != cond:
                    return False
        return True

    def query(self, vector: list, top_k: int = 5, namespace: str = "",
              filter: dict | None = None, include_metadata: bool = False) -> dict:
        ns = self._store.get(namespace, {})
        q = np.array(vector)
        scored = []
        for doc_id, entry in ns.items():
            if not self._matches_filter(entry["metadata"], filter):
                continue
            v = np.array(entry["values"])
            sim = float(np.dot(q, v) / (np.linalg.norm(q) * np.linalg.norm(v) + 1e-9))
            scored.append((doc_id, sim, entry["metadata"]))
        scored.sort(key=lambda t: t[1], reverse=True)
        matches = [
            {"id": doc_id, "score": score, **({"metadata": md} if include_metadata else {})}
            for doc_id, score, md in scored[:top_k]
        ]
        return {"matches": matches, "namespace": namespace}


# pc.create_index(...) equivalent -- just instantiate with the embedding dimension
index = FakePineconeIndex(dimension=EMBED_DIM, metric="cosine")
print("Fake index created, dimension:", index.dimension)

Fake index created, dimension: 32


## 3. Upsert project-data chunks, namespaced per client

Mirrors Chapter 2's multi-tenancy design: `namespace=client_id` is the hard isolation boundary.

In [4]:
acme_chunks = [
    {"id": "atlas-status-01", "text": "Project Atlas Japan localization on track for Q3 launch",
     "metadata": {"client_id": "acme-pharma", "project_id": "atlas", "doc_type": "status_update"}},
    {"id": "atlas-status-02", "text": "Project Atlas France localization delayed two weeks legal review",
     "metadata": {"client_id": "acme-pharma", "project_id": "atlas", "doc_type": "status_update"}},
    {"id": "atlas-cost-01", "text": "SKU-LOC-JP-STD Japanese localization standard package pricing",
     "metadata": {"client_id": "acme-pharma", "project_id": "atlas", "doc_type": "cost_catalog"}},
]

globex_chunks = [
    {"id": "orion-status-01", "text": "Project Orion EU regulatory submission passed compliance review",
     "metadata": {"client_id": "globex-bio", "project_id": "orion", "doc_type": "status_update"}},
]

for chunk in acme_chunks:
    index.upsert(
        vectors=[{"id": chunk["id"], "values": fake_embed(chunk["text"]), "metadata": chunk["metadata"]}],
        namespace="acme-pharma",
    )

for chunk in globex_chunks:
    index.upsert(
        vectors=[{"id": chunk["id"], "values": fake_embed(chunk["text"]), "metadata": chunk["metadata"]}],
        namespace="globex-bio",
    )

print("Upserted", len(acme_chunks), "chunks for acme-pharma and", len(globex_chunks), "for globex-bio")

Upserted 3 chunks for acme-pharma and 1 for globex-bio


## 4. Query with namespace + metadata filter

The query below asks about "Project Atlas status" scoped to `acme-pharma`'s namespace, further
filtered to `doc_type=status_update` -- exactly the `retrieve_for_client` pattern from Chapter 2.

In [5]:
def retrieve_for_client(query_text: str, client_id: str, project_id: str | None = None,
                          doc_type: str | None = None, top_k: int = 5):
    filt = {}
    if project_id:
        filt["project_id"] = {"$eq": project_id}
    if doc_type:
        filt["doc_type"] = {"$eq": doc_type}
    return index.query(
        vector=fake_embed(query_text),
        top_k=top_k,
        namespace=client_id,       # hard isolation boundary -- never user-supplied
        filter=filt,               # soft, query-specific narrowing
        include_metadata=True,
    )

result = retrieve_for_client(
    "What is the status of the France localization?",
    client_id="acme-pharma",
    project_id="atlas",
    doc_type="status_update",
)
result["matches"]

[{'id': 'atlas-status-02',
  'score': 0.35634832219355084,
  'metadata': {'client_id': 'acme-pharma',
   'project_id': 'atlas',
   'doc_type': 'status_update'}},
 {'id': 'atlas-status-01',
  'score': 0.08908708054838771,
  'metadata': {'client_id': 'acme-pharma',
   'project_id': 'atlas',
   'doc_type': 'status_update'}}]

## 5. Proving isolation: `globex-bio` never sees `acme-pharma` data

Query the same text but scoped to `globex-bio`'s namespace -- the Atlas chunks (which only exist
under `acme-pharma`) cannot appear, no matter how semantically similar the query is.

In [6]:
cross_tenant_result = retrieve_for_client(
    "What is the status of the France localization for Project Atlas?",
    client_id="globex-bio",
)
print("Matches visible to globex-bio:", cross_tenant_result["matches"])
assert all(m["id"] != "atlas-status-02" for m in cross_tenant_result["matches"]), \
    "Isolation violated -- acme-pharma data leaked into globex-bio's namespace!"
print("\nIsolation check passed: no acme-pharma chunks visible in globex-bio's namespace.")

Matches visible to globex-bio: [{'id': 'orion-status-01', 'score': 0.5368754916562838, 'metadata': {'client_id': 'globex-bio', 'project_id': 'orion', 'doc_type': 'status_update'}}]

Isolation check passed: no acme-pharma chunks visible in globex-bio's namespace.


## Takeaways

- `FakePineconeIndex.upsert` / `.query` mirror the real `pinecone.Index` method signatures closely
  enough that swapping in the real client is a constructor-line change, not a rewrite of calling code.
- Namespace-per-client is what makes the isolation assertion in cell 5 provable: a query against one
  namespace structurally cannot return another namespace's vectors, regardless of query content.
- Metadata filters (`project_id`, `doc_type`) narrow *within* a namespace and are applied alongside
  the similarity search, not as a slow post-filter -- see `../02-vector-databases-and-pinecone.md`
  for why that ordering matters at scale.